### Tutorial 2 — Step-by-step patch feature extraction with CORAL

CORAL is **patch-centric**: the primary unit of analysis is a patch.

This tutorial continues from the slide you ingested in [Tutorial 1](1-Tissue-Ingest.ipynb) and walks it through the patch feature-extraction pipeline:

1. **Tissue segmentation** — tissue vs. background
2. **Patch extraction** — patch coordinates, each tagged with its tissue fraction
3. **Feature extraction** — encode each patch (mean-marker baseline **and** KRONOS2)

> **Prerequisites.** Run [Tutorial 0](0-Example-Data-Download.ipynb) (downloads the data) and [Tutorial 1](1-Tissue-Ingest.ipynb) (ingest) first — this notebook reopens the slide store Tutorial 1 wrote. **Launch Jupyter from the `tutorials/` directory**, and run the `coral` commands below from a terminal at the repo root (`CORAL/`).

Each step is driven by an **`coral` CLI command** (run in a terminal), and the notebook displays its result. The CLI runs each step over a **whole cohort** — every `.zarr` store in a **job directory** (here `processed/`, the folder Tutorial 1 wrote `raw_image.zarr` into).

Similar to the previous tutorial, a final section shows the equivalent **Python** (`CoralSlide`) API for scripting on a single slide.

| Step     | CLI           | Python                          |
| -------- | ------------- | ------------------------------- |
| Tissue   | `coral tissue`  | `slide.detect_tissue(...)`      |
| Patches  | `coral patch`   | `slide.extract_patches(...)`    |
| Features | `coral extract` | `slide.encode_features(...)`    |
| Status   | `coral status`  | `slide.status()`                |

#### 0 — Installation

```bash
# Base install — the mean-marker encoder works out of the box (pure NumPy)
uv sync

# Optional: KRONOS2 foundation-model features (needs a CUDA GPU)
uv sync --extra kronos2
```

Only `mean_marker` is available with the base install; the foundation-model
encoders and Cellpose are opt-in extras because they pull in heavy,
version-pinned ML stacks.

#### Reopen the slide from Tutorial 1

This notebook continues from the slide [Tutorial 1](1-Tissue-Ingest.ipynb) ingested — no re-ingest. Tutorial 1 named that store after its input folder (`raw_image/` → `raw_image.zarr`), so we point `CoralSlide.open` straight at it. Reopening the slide is Python-only; there's no cohort command for it.

In [ ]:
from pathlib import Path

from coral import CoralSlide

# Run this notebook from the tutorials/ directory (Jupyter's working dir).
# Tutorial 1 ingested the slide to processed/raw_image.zarr; reopen it.
OUTPUT_DIR = Path("example-data/processed")
PATCH_SIZE = 256              # 256 px is ~95 um at 0.37 mpp
SLUG = "0.37mpp_256px"        # patch-set id (<mpp>mpp_<size>px), from coral patch

store_path = OUTPUT_DIR / "raw_image.zarr"
assert store_path.exists(), (
    f"No ingested slide at {store_path}. "
    "Run Tutorial 1 (1-Tissue-Ingest.ipynb) first."
)
slide = CoralSlide.open(store_path)
print(slide)

#### 1. Tissue vs. background segmentation

`coral tissue` runs a segmenter over the nuclear + structural channels at full
resolution and writes a per-method `tissue/tissue_<method>/` folder into each
store — here `tissue/tissue_otsu/`:

- `tissue/tissue_otsu/tissue.geojson` — the boundary polygons (**source of
  truth**; you can hand-edit this in QuPath and downstream steps respect it)
- `tissue/tissue_otsu/tissue_mask.png` — the binary mask
- `tissue/tissue_otsu/tissue_overlay.png` — a review overlay of the boundary on
  the nuclear channel

`otsu` is the default method; `carta` (nuclear-only) is also built in. Pick one
with `--segmentation-method`, and the output subfolder is named after it.

**Run it — tissue over the cohort.** Otsu tissue detection on every store in the job directory:

```bash
uv run coral tissue --job-dir tutorials/example-data/processed
```

Choose which structural markers feed the composite with `--structural-markers vimentin,...`, or import a hand-drawn mask with `--custom-mask-path`.

In [ ]:
from PIL import Image
from IPython.display import display

# coral tissue wrote a review overlay into the store, under the method's subfolder.
display(Image.open(slide.path / "tissue" / "tissue_otsu" / "tissue_overlay.png"))

> Examine the overlay before trusting downstream patches — the boundary should track the tissue edges. If coverage looks degenerate (all-tissue or all-background), CORAL logs a warning; re-run with different `--structural-markers` or a hand-drawn `--custom-mask-path`.

#### 2. Patch coordinate extraction

`coral patch` tiles the slide into patches at the base resolution, **keeping every patch** and recording each patch's tissue fraction alongside its coordinates — so you filter by tissue later, at analysis time, rather than discarding patches at extraction. It writes `patches/<slug>/` (coordinates, per-patch tissue fraction, config, and a review overlay). The **slug** encodes mpp + size (e.g. `0.37mpp_256px`), so several patch sets can coexist on one slide without confusion. Add `--overlap 0.5` for 50%-overlapping patches.

**Run it — patches over the cohort.** A patch grid on each store (`grid` is the default mode):

```bash
uv run coral patch --job-dir tutorials/example-data/processed --patch-size 256
```

In [ ]:
# coral patch wrote patches/<slug>/patch_overlay.png — every patch over the tissue
# contour, each labeled by its tissue fraction.
display(Image.open(slide.path / "patches" / SLUG / "patch_overlay.png"))

#### 3. Patch feature extraction

`coral extract` reads a patch set, materializes each patch's pixels, runs an
**encoder**, and writes the per-patch features under
`features/<slug>/<encoder>/<variant>/`.

We start with `mean_marker` — the per-marker mean intensity of each patch. It's
pure NumPy (no GPU, no weights), which makes it the ideal baseline and sanity
check. It produces an `(n_patches, n_markers)` array.

**Run it — features over the cohort.** The mean-marker baseline (CPU, no weights), selecting the patch set by its slug:

```bash
uv run coral extract --job-dir tutorials/example-data/processed --extractor mean_marker --patches 0.37mpp_256px
```

Omit `--patches` to encode every patch set on each store.

In [ ]:
# Read the features coral extract wrote — a labeled xarray.Dataset with named
# dims (patch, marker) and coordinates (marker names + each patch's x/y).
# features() takes the encoder name and the slug as plain strings.
ds = slide.features("mean_marker", SLUG)
ds

##### Upgrading to a foundation model (KRONOS2)

Swapping the encoder is a one-flag change — every encoder in the registry plugs into the same `coral extract` call. `KRONOS2` gives a 768-dim embedding per patch. This needs the `kronos2` extra, a CUDA GPU (see step 0), **and** granted access to the KRONOS2 Hugging Face repo:

```bash
uv run coral extract --job-dir tutorials/example-data/processed --extractor KRONOS2 --patches 0.37mpp_256px --batch-size 16 --gpu 0
```

> On RTX 3090, batch size of 16 patches with 49 markers takes around ~15 minutes, with peak GPU usage of ~11GB.

Read the embeddings back the same way — `slide.features("KRONOS2", "0.37mpp_256px")` → an xarray with dims `(patch, feature)`. Check progress any time with `uv run coral status --job-dir tutorials/example-data/processed`.

> CORAL supports many more patch encoders than the ones shown here. Please refer to the main README.md for the latest collections of available patch encoders.

#### 4. The same pipeline in Python

Everything above ran through the `coral` CLI. The `CoralSlide` API does the *same* steps on a single slide, in-process — reach for it to script custom flows or work interactively. Each CLI step maps to one method call.

```python
from pathlib import Path

from coral import CoralSlide
from coral.config import PatchConfig
from coral.features import Kronos2Extractor, MeanMarkerExtractor
from coral.tissue import OtsuTissueSegmenter

slide = CoralSlide.open(Path("example-data/processed/raw_image.zarr"))

# 1. Tissue — segment tissue vs. background.
slide.detect_tissue(OtsuTissueSegmenter())

# 2. Patches — grid; every patch kept with its tissue fraction. Slug = mpp + size.
cfg = PatchConfig(patch_size=256, overlap=0.0)
slide.extract_patches(cfg)

# 3a. Features — mean-marker baseline (CPU, no weights).
slide.encode_features(MeanMarkerExtractor(), cfg)
ds = slide.features(MeanMarkerExtractor(), cfg)   # xarray (patch, marker)

# 3b. Upgrade the encoder — KRONOS2 768-dim embeddings (GPU + kronos2 extra).
kronos = Kronos2Extractor.from_pretrained()       # loads weights once
slide.encode_features(kronos, cfg, batch_size=16)
emb = slide.features(kronos, cfg)                 # xarray (patch, feature)
```

#### 5. Recap

You took one raw image all the way to patch embeddings, driven entirely from the command line. CORAL tracked every step in the slide's state file:

In [ ]:
{step: info["status"] for step, info in slide.status().items()}

**Where next?**
- **Cells** are a *sibling* capability (not a prerequisite) — see [Tutorial 3](3-Cell-Segmentation-and-Feature-Extraction.ipynb): `slide.segment_cells(CellposeSegmenter())` with the `cells` extra, then cell-centered patches via `PatchConfig(mode="cell_centered")`.
- **Patch clustering / phenotyping** — [Tutorial 4](4-Patch-Clustering.ipynb) clusters these KRONOS2 embeddings into unsupervised tissue domains.
- **Cohorts**: point the `coral` CLI at a directory of slides, or drive `CoralProcessor` from Python, for parallel, resumable, per-slide-tracked runs.
- **Custom encoders**: register your own model in `EXTRACTOR_REGISTRY` and use it exactly like the built-ins.